In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ========================================
# INSTALL DEPENDENCIES
# ========================================
!pip install -q torch transformers scikit-learn tqdm

In [ ]:
# ========================================
# IMPORTS AND DEVICE SETUP
# ========================================
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda
Dataset Info:
Shape: (223549, 11)
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'input_word_ids', 'input_mask', 'all_segment_id']


In [ ]:
# ========================================
# DATA PREPROCESSING
# ========================================
text_column = 'comment_text'

# Create binary toxic label
toxicity_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
available_toxicity_cols = [col for col in toxicity_columns if col in df.columns]

if len(available_toxicity_cols) > 0:
    df['is_toxic'] = df[available_toxicity_cols].max(axis=1)
else:
    df['is_toxic'] = df[df.columns[1]]

# Clean text
df[text_column] = df[text_column].fillna('')
df[text_column] = df[text_column].astype(str)

# Sample for faster training
sample_size = 50000  # Use len(df) for full dataset
if len(df) > sample_size:
    df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    print(f"Using {sample_size} samples")

# Split data
X = df[text_column].values
y = df['is_toxic'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")
print(f"Toxic ratio: {y_train.mean():.2%}")

Using 50000 samples

Training samples: 40000
Validation samples: 5000
Test samples: 5000
Toxic ratio: 10.00%


In [ ]:
# ========================================
# LOAD TOKENIZER
# ========================================
tokenizer = DistilBertTokenizer.from_pretrained(
    'distilbert-base-uncased'
)

print("✓ Tokenizer loaded!")

In [ ]:
# ========================================
# PYTORCH DATASET AND DATALOADERS
# ========================================
class ToxicCommentsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


train_dataset = ToxicCommentsDataset(X_train, y_train, tokenizer)
val_dataset = ToxicCommentsDataset(X_val, y_val, tokenizer)
test_dataset = ToxicCommentsDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("✓ Data loaders created successfully!")

NameError: name 'tokenizer' is not defined

In [ ]:
# ========================================
# LOAD PRETRAINED DISTILBERT MODEL
# ========================================
print("Loading DistilBERT model...")

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    eps=1e-8
)

epochs = 3
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print("✓ Model loaded and ready!")


Loading DistilBERT model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded and ready!


In [ ]:
# ========================================
# TRAINING FUNCTION
# ========================================
def train_epoch(model, data_loader, optimizer, device, scheduler):
    model.train()
    losses = []
    correct_predictions = 0

    progress_bar = tqdm(data_loader, desc='Training')

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

        losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        progress_bar.set_postfix({'loss': np.mean(losses)})

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

def eval_model(model, data_loader, device):
    model.eval()
    losses = []
    correct_predictions = 0

    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

In [ ]:
# ========================================
# TRAIN THE MODEL
# ========================================
MODEL_DIR = "/content/drive/MyDrive/DL_Projects/model"
BEST_MODEL_PATH = f"{MODEL_DIR}/best_distilbert_toxic.pth"

print("Starting training...")
print("=" * 60)

best_val_loss = float('inf')
history = {
    'train_acc': [],
    'train_loss': [],
    'val_acc': [],
    'val_loss': []
}

for epoch in range(epochs):
    print(f'\nEpoch {epoch + 1}/{epochs}')
    print('-' * 60)

    train_acc, train_loss = train_epoch(
        model, train_loader, optimizer, device, scheduler
    )

    print(f'Train loss: {train_loss:.4f} | Train accuracy: {train_acc:.4f}')

    val_acc, val_loss = eval_model(
        model, val_loader, device
    )

    print(f'Val loss: {val_loss:.4f} | Val accuracy: {val_acc:.4f}')

    history['train_acc'].append(train_acc.cpu().numpy())
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc.cpu().numpy())
    history['val_loss'].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print('✓ Best model saved!')

print("\n" + "=" * 60)
print("Training complete!")
print("=" * 60)

Starting training...

Epoch 1/3
------------------------------------------------------------


Training: 100%|██████████| 2500/2500 [07:43<00:00,  5.40it/s, loss=0.136]


Train loss: 0.1362 | Train accuracy: 0.9511


Evaluating: 100%|██████████| 313/313 [00:19<00:00, 15.76it/s]


Val loss: 0.1247 | Val accuracy: 0.9590
✓ Model saved!

Epoch 2/3
------------------------------------------------------------


...

Val loss: 0.1315 | Val accuracy: 0.9592

Epoch 3/3
------------------------------------------------------------


Training: 100%|██████████| 2500/2500 [07:46<00:00,  5.36it/s, loss=0.0498]


Train loss: 0.0498 | Train accuracy: 0.9858


Evaluating: 100%|██████████| 313/313 [00:20<00:00, 15.62it/s]

Val loss: 0.1800 | Val accuracy: 0.9572

Training complete!


In [ ]:
# ========================================
# EVALUATE ON TEST SET
# ========================================
TOKENIZER_PATH = f"{MODEL_DIR}/distilbert_tokenizer"

# Load best model
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

model = model.to(device)

print("Evaluating on test set...")

test_acc, test_loss = eval_model(
    model,
    test_loader,
    device
)

print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")
print("=" * 60)

# Save tokenizer
tokenizer.save_pretrained(TOKENIZER_PATH)

print("\n✓ Tokenizer saved!")

NameError: name 'model' is not defined

In [ ]:
# ========================================
# TEXT TOXICITY PREDICTION
# ========================================
def predict_text_toxic(text, model, tokenizer, device):
    """Return the probability that the text is toxic."""

    if not text or len(text.strip()) == 0:
        return 0.0

    model.eval()

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(outputs.logits, dim=1)
        toxic_probability = probabilities[0][1].item()

    return toxic_probability

Testing predictions:
Text: 'You are stupid and worthless'
Score: 0.9829 - TOXIC
------------------------------------------------------------
Text: 'Have a great day!'
Score: 0.0018 - NOT TOXIC
------------------------------------------------------------
Text: 'I hate you so much'
Score: 0.9479 - TOXIC
------------------------------------------------------------
Text: 'This is a nice picture'
Score: 0.0007 - NOT TOXIC
------------------------------------------------------------


In [ ]:
# ========================================
# TEST TEXT PREDICTIONS
# ========================================
test_texts = [
    "You are stupid and worthless",
    "Have a great day!",
    "I hate you so much",
    "This is a nice picture"
]

print("Testing predictions:")
print("=" * 60)

for text in test_texts:
    prediction = predict_text_toxic(
        text,
        model,
        tokenizer,
        device
    )

    label = "TOXIC" if prediction > 0.5 else "NOT TOXIC"

    print(f"Text: '{text}'")
    print(f"Score: {prediction:.4f} - {label}")
    print("-" * 60)